# Analyse nicht-gematchter HD-Wörter
Warum wurden bestimmte HD-Wörter vom Greedy-Alignment nicht gematch?

Kategorien:
- **Gematch** – erfolgreich im Mapping
- **Zu wenig Hits** – bester IPA-Kandidat unterschreitet `MIN_HITS`
- **Rate zu tief** – bester Kandidat unter `MIN_KOKKURRENZ`
- **PMI zu tief** – bester Kandidat unter `MIN_PMI`
- **Greedy verdrängt** – gute Scores, aber bester IPA-Token bereits von anderem HD-Wort beansprucht

In [ ]:
import math
import re
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({"figure.dpi": 130, "font.size": 11})

# Gleiche Parameter wie build_mapping.py
MIN_HITS       = 4
MIN_KOKKURRENZ = 0.0001
MIN_PMI        = 0.0

df_raw     = pd.read_csv("Data/transcriptions_tenses.csv")
df_ost     = df_raw[df_raw["dialect_region"] == "Ostschweiz"].copy().reset_index(drop=True)
df_matched = pd.read_csv("Data/ostschweiz_mapping_results.csv")

matched_words = set(df_matched["Hochdeutsch"])
matched_ipa   = set(df_matched["IPA_Dialekt"])

print(f"Sätze:          {len(df_ost):,}")
print(f"Gematchte Paare: {len(df_matched)}")
print(f"Gematchte HD-Wörter: {len(matched_words)}")

In [ ]:
def clean_hd(text):
    return re.sub(r"[^\w\s]", "", str(text).lower()).split()

def tokenize_ipa(text):
    return str(text).strip().split()

# Corpus-Frequenzen und Zeilen-Index pro HD-Wort
corpus_freq  = Counter()
hd_word_rows = defaultdict(set)
for idx, row in df_ost.iterrows():
    for word in clean_hd(row["sentence"]):
        corpus_freq[word] += 1
        hd_word_rows[word].add(idx)

# IPA-Clip-Häufigkeiten (wie viele Clips enthält jedes IPA-Token)
n_clips = len(df_ost)
ipa_clip_counts = defaultdict(int)
for ipa_text in df_ost["ipa_audio"]:
    for tok in set(tokenize_ipa(ipa_text)):
        ipa_clip_counts[tok] += 1

print(f"Einzigartige HD-Wörter: {len(corpus_freq):,}")
print(f"Einzigartige IPA-Token: {len(ipa_clip_counts):,}")

In [ ]:
# Für jedes HD-Wort: besten IPA-Kandidaten und seine Scores berechnen (Single-Pass)
records = []

for hd_word, indices in hd_word_rows.items():
    clip_count = len(indices)
    ipa_counts = defaultdict(int)
    for idx in indices:
        for tok in set(tokenize_ipa(df_ost.loc[idx, "ipa_audio"])):
            ipa_counts[tok] += 1

    if not ipa_counts:
        records.append({"Hochdeutsch": hd_word, "Clips": clip_count,
                        "BestIPA": None, "BestHits": 0,
                        "BestRate": 0.0, "BestPMI": None})
        continue

    best_tok, best_hits = max(ipa_counts.items(), key=lambda x: x[1])
    best_rate = best_hits / clip_count
    best_pmi  = math.log2((best_hits * n_clips) /
                          (clip_count * ipa_clip_counts[best_tok])) if best_hits > 0 else None

    records.append({
        "Hochdeutsch": hd_word,
        "Clips":       clip_count,
        "BestIPA":     best_tok,
        "BestHits":    best_hits,
        "BestRate":    round(best_rate, 4),
        "BestPMI":     round(best_pmi, 4) if best_pmi is not None else None,
    })

df_all = pd.DataFrame(records)
print(f"HD-Wörter total: {len(df_all):,}")
df_all.head()

In [ ]:
# Kategorie zuweisen
def kategorisieren(row):
    if row["Hochdeutsch"] in matched_words:
        return "Gematch"
    if row["BestHits"] < MIN_HITS:
        return "Zu wenig Hits"
    if row["BestRate"] < MIN_KOKKURRENZ:
        return "Rate zu tief"
    if row["BestPMI"] is None or row["BestPMI"] < MIN_PMI:
        return "PMI zu tief"
    if row["BestIPA"] in matched_ipa:
        return "Greedy verdrängt"
    return "Unbekannt"

df_all["Kategorie"] = df_all.apply(kategorisieren, axis=1)

counts = df_all["Kategorie"].value_counts()
print("Kategorienverteilung:")
for cat, n in counts.items():
    print(f"  {cat:<25} {n:>5}  ({n/len(df_all)*100:.1f}%)")

In [ ]:
# Balkendiagramm Kategorien
colors = {
    "Gematch":           "#27AE60",
    "Zu wenig Hits":     "#AED6F1",
    "Rate zu tief":      "#E74C3C",
    "PMI zu tief":       "#E67E22",
    "Greedy verdrängt":  "#8E44AD",
    "Unbekannt":         "#95A5A6",
}

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(
    counts.index,
    counts.values,
    color=[colors.get(c, "#95A5A6") for c in counts.index]
)
ax.bar_label(bars, padding=4, fontsize=9)
ax.set_xlabel("Anzahl HD-Wörter")
ax.set_title("Warum wurden HD-Wörter nicht gematch?")
plt.tight_layout()
plt.show()

In [ ]:
# Scatter: Rate vs PMI – alle Wörter mit >= MIN_HITS, eingefärbt nach Kategorie
df_plot = df_all[(df_all["BestHits"] >= MIN_HITS) & df_all["BestPMI"].notna()].copy()

fig, ax = plt.subplots(figsize=(10, 6))
for cat, grp in df_plot.groupby("Kategorie"):
    ax.scatter(grp["BestRate"], grp["BestPMI"],
               label=cat, alpha=0.6, s=20, color=colors.get(cat, "grey"))

ax.axvline(MIN_KOKKURRENZ, color="red",   linestyle="--", linewidth=0.8, label=f"MIN_KOKKURRENZ={MIN_KOKKURRENZ}")
ax.axhline(MIN_PMI,        color="orange", linestyle="--", linewidth=0.8, label=f"MIN_PMI={MIN_PMI}")
ax.set_xlabel("Beste Kokkurrenz_Rate")
ax.set_ylabel("Bester PMI")
ax.set_title("Rate vs. PMI aller HD-Wörter (BestHits ≥ MIN_HITS)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Tabelle: «Fast gematch» – hohe Scores aber verdrängt oder gerade unter Schwelle
df_almost = (
    df_all[
        (df_all["Kategorie"].isin(["Greedy verdrängt", "PMI zu tief", "Rate zu tief"])) &
        (df_all["BestHits"] >= MIN_HITS)
    ]
    .sort_values("BestPMI", ascending=False)
    .head(30)
    .reset_index(drop=True)
)
print("Top-30 'Fast gematch':")
print(df_almost[["Hochdeutsch", "Clips", "BestIPA", "BestHits", "BestRate", "BestPMI", "Kategorie"]].to_string(index=False))